In [ ]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import seaborn as sns
import json
from collections import defaultdict
load_dotenv()

questions_path=os.getenv("QUESTIONS_PATH")
t_df=pd.read_json(questions_path)
question_df=t_df.transpose()
question_df=question_df.reset_index()

class KCTree:
    def __init__(self):
        self.tree = {}
        self.leaf_to_path = {}
        self.path_to_leaf = {}
        self.all_nodes = set()
    
    def add_path(self, path):
        """Add a KC path to the tree"""
        if '----' not in path:
            return
        
        parts = path.split('----')
        current = self.tree
        
        # Build the path in the tree
        for i, part in enumerate(parts):
            if part not in current:
                current[part] = {}
            current = current[part]
            
            # Track all nodes at each level
            self.all_nodes.add(part)
            
            # If it's a leaf (last part), store the mapping
            if i == len(parts) - 1:
                self.leaf_to_path[part] = path
                self.path_to_leaf[path] = part
    
    def build_from_dataframe(self, df, column='kc_routes'):
        """Build tree from all routes in dataframe"""
        for routes in df[column]:
            if isinstance(routes, list):
                for path in routes:
                    self.add_path(path)
            elif isinstance(routes, str) and '----' in routes:
                self.add_path(routes)
    
    def get_tree(self):
        """Return the tree dictionary"""
        return self.tree
    
    def get_all_levels(self):
        """Get all unique nodes at each level"""
        levels = defaultdict(set)
        for path in self.path_to_leaf.keys():
            parts = path.split('----')
            for i, part in enumerate(parts):
                levels[i].add(part)
        return {level: list(nodes) for level, nodes in sorted(levels.items())}
    
    def get_parents(self, leaf_name):
        """Get the parent hierarchy for a leaf"""
        if leaf_name in self.leaf_to_path:
            path = self.leaf_to_path[leaf_name]
            parts = path.split('----')
            return parts[:-1]  # Everything except the leaf
        return []
    
    def get_children(self, node_name):
        """Get all children of a specific node"""
        def find_node(tree, target, current_path=None):
            if current_path is None:
                current_path = []
            
            for key, value in tree.items():
                new_path = current_path + [key]
                if key == target:
                    return value
                result = find_node(value, target, new_path)
                if result is not None:
                    return result
            return None
        
        node = find_node(self.tree, node_name)
        if node is not None:
            return list(node.keys())
        return []
    
    def get_depth(self, node_name):
        """Get the depth of a node in the tree"""
        for path in self.path_to_leaf.keys():
            parts = path.split('----')
            if node_name in parts:
                return parts.index(node_name)
        return -1
    
    def get_cluster_levels(self):
        """Get nodes that could serve as cluster labels at different levels"""
        levels = self.get_all_levels()
        
        cluster_levels = {}
        for level, nodes in levels.items():
            # Get the parent context for each node at this level
            cluster_levels[level] = {
                'nodes': nodes,
                'count': len(nodes),
                'depth': level + 1
            }
        return cluster_levels
    
    def print_tree(self, tree=None, indent=0, max_depth=None):
        """Pretty print the tree"""
        if tree is None:
            tree = self.tree
        
        for key, value in sorted(tree.items()):
            print("  " * indent + key)
            if max_depth is None or indent < max_depth:
                self.print_tree(value, indent + 1, max_depth)

# Build the tree
kc_tree = KCTree()
kc_tree.build_from_dataframe(question_df)

with open('kc_tree_structure.md', 'w', encoding='utf-8') as f:
    f.write("# KC Tree Structure\n\n")
    def write_markdown_tree(tree, file, indent=0, prefix=""):
        keys = sorted(tree.keys())
        for i, key in enumerate(keys):
            is_last = i == len(keys) - 1
            connector = "└── " if is_last else "├── "
            file.write("    " * indent + connector + key + "\n")
            
            if tree[key]:
                new_prefix = "    " * indent + ("    " if is_last else "│   ")
                write_markdown_tree(tree[key], file, indent + 1)
    
    f.write("## Tree Structure\n\n```\n")
    write_markdown_tree(kc_tree.tree, f)
    f.write("```\n\n")
    
    f.write("## Cluster Levels\n\n")
    f.write("| Level | Depth | Number of Clusters | Example Nodes |\n")
    f.write("|-------|-------|-------------------|---------------|\n")
    
    for level, info in kc_tree.get_cluster_levels().items():
        depth = level + 1
        examples = ", ".join(info['nodes'][:5])
        f.write(f"| {depth} | {depth} | {info['count']} | {examples} |\n")
    
    f.write("\n## Statistics\n\n")
    f.write(f"- **Total unique nodes**: {len(kc_tree.all_nodes)}\n")
    f.write(f"- **Total leaf nodes**: {len(kc_tree.leaf_to_path)}\n")
    f.write(f"- **Maximum depth**: {max(kc_tree.get_all_levels().keys()) + 1}\n")

print("Markdown file saved: kc_tree_structure.md")